# Notebook 04: Figures

Generate all figures for the paper:
- **Figure 3**: Metric vs Runtime scatter plots (log-log)
- **Figure 4**: Predicted vs Actual runtime
- **Figure 5**: OOS residuals boxplot

In [ ]:
import sys
import os
import json
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('../src'))
from utils import (load_metrics, load_runtime, merge_metrics_runtime,
                    fit_loglog, predict_loglog)

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Paper-quality figure settings
plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

print('Setup complete.')

## 1. Load all data

In [ ]:
# Load metrics — prefer notebook 01 output, fall back to pre-computed
computed_path = OUTPUT_DIR / 'computed_metrics.json'
if computed_path.exists():
    raw = load_metrics(computed_path)
    # computed_metrics.json is {"models": [...]}, with "name" instead of "model_name"
    if isinstance(raw, dict) and 'models' in raw:
        metrics = raw['models']
        for m in metrics:
            if 'model_name' not in m and 'name' in m:
                m['model_name'] = m['name']
    else:
        metrics = raw
    print(f'Loaded metrics from notebook 01 output ({len(metrics)} models)')
else:
    metrics = load_metrics(DATA_DIR / 'results' / 'all_metrics.json')
    print(f'Loaded pre-computed metrics ({len(metrics)} models)')

runtime = load_runtime(DATA_DIR / 'results' / 'hybrid_runtime.json')
merged = merge_metrics_runtime(metrics, runtime)

dev_set = [m for m in merged if m['topology'] in ('tandem', 'fork_join', 'feedback')]

# OOS: 3x3 factorial design (N in {2,3,4}, M in {2,3,4}) to match the paper
oos_factorial = {f'hybrid_{n}_{m}' for n in (2, 3, 4) for m in (2, 3, 4)}
oos_set = [m for m in merged if m['model_name'] in oos_factorial]

# Use only dev + OOS factorial for all figures
plot_set = dev_set + oos_set

# Topology markers and colors
TOPO_STYLE = {
    'tandem':    {'marker': 'o', 'color': '#1f77b4', 'label': 'Tandem'},
    'fork_join': {'marker': 's', 'color': '#ff7f0e', 'label': 'Fork-Join'},
    'feedback':  {'marker': '^', 'color': '#2ca02c', 'label': 'Feedback'},
    'hybrid':    {'marker': 'D', 'color': '#d62728', 'label': 'Hybrid (OOS)'},
}

print(f'Dev: {len(dev_set)}, OOS: {len(oos_set)}, Plot total: {len(plot_set)}')

## 2. Figure 3: Metric vs Runtime (log-log, 4 panels)

In [ ]:
metric_configs = [
    ('SMC', 'smc', 'SMC (SimASM Model Complexity)'),
    ('CC', 'cyclomatic_number', 'CC (Cyclomatic Number)'),
    ('LOC', 'loc', 'LOC (Lines of Code)'),
    ('KC', 'kc', 'KC (Kolmogorov Complexity)'),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for idx, (short_name, field, xlabel) in enumerate(metric_configs):
    ax = axes[idx]
    
    # Plot each topology (using plot_set = dev + OOS factorial only)
    for topo, style in TOPO_STYLE.items():
        subset = [m for m in plot_set if m['topology'] == topo]
        if not subset:
            continue
        x = [m[field] for m in subset]
        y = [m['wall_time'] for m in subset]
        ax.scatter(x, y, marker=style['marker'], color=style['color'],
                   label=style['label'], s=40, alpha=0.8, edgecolors='white', linewidth=0.5)
    
    # Fit line on dev set only
    dev_x = np.array([m[field] for m in dev_set], dtype=float)
    dev_y = np.array([m['wall_time'] for m in dev_set], dtype=float)
    fr = fit_loglog(dev_x, dev_y)
    
    # Plot regression line
    x_range = np.linspace(min(dev_x.min(), min(m[field] for m in oos_set)) * 0.9,
                          max(dev_x.max(), max(m[field] for m in oos_set)) * 1.1, 100)
    y_pred = predict_loglog(x_range, fr['slope'], fr['intercept'])
    ax.plot(x_range, y_pred, 'k--', alpha=0.5, linewidth=1,
            label=f'Fit: R²={fr["r2"]:.3f}')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Runtime (s)')
    ax.legend(fontsize=8)
    ax.set_title(f'({chr(97+idx)}) {short_name} vs Runtime')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_3_metric_vs_runtime.pdf')
fig.savefig(OUTPUT_DIR / 'figure_3_metric_vs_runtime.png')
plt.show()
print('Saved Figure 3')

## 3. Figure 4: Predicted vs Actual (4 panels)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for idx, (short_name, field, xlabel) in enumerate(metric_configs):
    ax = axes[idx]
    
    # Fit on dev set
    dev_x = np.array([m[field] for m in dev_set], dtype=float)
    dev_y = np.array([m['wall_time'] for m in dev_set], dtype=float)
    fr = fit_loglog(dev_x, dev_y)
    
    # Predict for dev + OOS factorial only (using plot_set)
    for topo, style in TOPO_STYLE.items():
        subset = [m for m in plot_set if m['topology'] == topo]
        if not subset:
            continue
        actual = np.array([m['wall_time'] for m in subset])
        x = np.array([m[field] for m in subset], dtype=float)
        predicted = predict_loglog(x, fr['slope'], fr['intercept'])
        ax.scatter(actual, predicted, marker=style['marker'], color=style['color'],
                   label=style['label'], s=40, alpha=0.8, edgecolors='white', linewidth=0.5)
    
    # Perfect prediction line
    all_rt = np.array([m['wall_time'] for m in plot_set])
    lim = [all_rt.min() * 0.7, all_rt.max() * 1.3]
    ax.plot(lim, lim, 'k-', alpha=0.3, linewidth=1, label='Perfect prediction')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Actual Runtime (s)')
    ax.set_ylabel('Predicted Runtime (s)')
    ax.legend(fontsize=8)
    ax.set_title(f'({chr(97+idx)}) {short_name}')
    ax.set_aspect('equal')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_4_predicted_vs_actual.pdf')
fig.savefig(OUTPUT_DIR / 'figure_4_predicted_vs_actual.png')
plt.show()
print('Saved Figure 4')